# 03 — Feature Engineering & Target Forward-Looking

> **Etapa 4A do Datathon FIAP Fase 5 — Associação Passos Mágicos**

## Objetivo

Preparar o dataset pra modelagem preditiva: definir o **target forward-looking**, criar **features derivadas** a partir dos insights das perguntas de negócio, e fazer o **split temporal anti-leakage**.

No final desta etapa, a gente vai ter 4 arquivos prontos pra Etapa 4B (modelagem):
- `X_train.parquet` e `y_train.parquet` — features e target de **2022 → 2023** (treino)
- `X_test.parquet` e `y_test.parquet` — features e target de **2023 → 2024** (teste)

## Decisões travadas

| # | Decisão |
|---|---|
| 1 | **Target composto (OU)**: queda INDE ≤ -0.5 OU regressão de pedra OU evasão OU inde < p25 da fase |
| 2 | **Split temporal**: treino 22→23 / teste 23→24 (anti-leakage por construção) |
| 3 | **Features derivadas**: gaps Dunning-Kruger, z-scores por fase, rankings |
| 4 | **IPP fora do modelo**: não existe em 2022, não pode ser feature do treino |

## Por que target forward-looking?
O objetivo da ONG é **identificar alunos em risco ANTES que eles caiam** — não depois. Por isso o modelo usa dados de t pra prever o estado em t+1. Isso é diferente de classificar o aluno como "em risco" no próprio ano (que seria apenas descritivo).

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 5)

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

alunos = pd.read_parquet(DATA_PROCESSED / "alunos_long.parquet")
print(f"✅ Dataset carregado: {alunos.shape}")
print(f"   Anos: {sorted(alunos['ano'].unique())}")

## 2. Construção do target forward-looking

O target `risco_tplus1` é **1** se, no ano t+1, QUALQUER uma destas condições for verdadeira:

| Regra | Condição | Interpretação |
|---|---|---|
| **(a)** | `delta_inde <= -0.5` | Queda significativa no INDE |
| **(b)** | Pedra regrediu (ex: Topázio→Ametista) | Perda de nível |
| **(c)** | Aluno ausente em t+1 (evasão) | Saiu do programa |
| **(d)** | `inde_tp1 < p25 da fase` | Ficou no quartil inferior da sua fase |

As 4 regras se sobrepõem (aluno que evadiu não tem INDE pra comparar, aluno que caiu de pedra geralmente caiu de INDE etc), mas a regra OU garante que capturamos **qualquer tipo de risco**, não apenas o mais óbvio.

**Inclusão de evasão como parte do target** é uma decisão pensada: depois da análise da P10, ficou claro que a evasão é o principal problema da Passos. Um modelo que só prevê "cai de INDE" ignoraria o risco mais impactante.

In [ ]:
ORDEM_PEDRAS = {"Quartzo": 0, "Ágata": 1, "Ametista": 2, "Topázio": 3}


def construir_target(df, ano_t, ano_tp1):
    """
    Constrói o target forward-looking: dado o ano t, prever se o aluno estará
    em risco em t+1.

    Returns:
        DataFrame com os dados de t + colunas de target e regras individuais.
    """
    # Todos os alunos presentes no ano t (base de features)
    em_t = df[df.ano == ano_t].copy()

    # Dados do ano t+1 (usado SÓ pra construir o target, nunca como feature!)
    em_tp1 = df[df.ano == ano_tp1][["ra", "inde", "pedra"]].copy()
    em_tp1.columns = ["ra", "inde_tp1", "pedra_tp1"]

    # LEFT JOIN preserva TODOS os alunos de t — evadidos terão NaN em tp1
    pares = em_t.merge(em_tp1, on="ra", how="left")

    # --- Regra (a): queda no INDE >= 0.5 pontos ---
    pares["delta_inde"] = pares["inde_tp1"] - pares["inde"]
    pares["r_queda"] = (pares["delta_inde"] <= -0.5).fillna(False)

    # --- Regra (b): regressão de pedra ---
    pares["nivel_t"] = pares["pedra"].map(ORDEM_PEDRAS)
    pares["nivel_tp1"] = pares["pedra_tp1"].map(ORDEM_PEDRAS)
    pares["r_regr"] = (pares["nivel_tp1"] < pares["nivel_t"]).fillna(False)

    # --- Regra (c): evasão (ausência em t+1) ---
    pares["r_evad"] = pares["inde_tp1"].isna() & pares["pedra_tp1"].isna()

    # --- Regra (d): INDE < p25 da fase em t+1 ---
    # O p25 é calculado APENAS com alunos que ficaram (não evadiram)
    p25_fase = (pares[~pares["r_evad"]]
                .groupby("fase")["inde_tp1"]
                .quantile(0.25)
                .to_dict())
    pares["inde_p25_fase"] = pares["fase"].map(p25_fase)
    pares["r_p25"] = (pares["inde_tp1"] < pares["inde_p25_fase"]).fillna(False)

    # --- Target composto (OR) ---
    pares["risco"] = (pares["r_queda"] | pares["r_regr"] |
                      pares["r_evad"] | pares["r_p25"]).astype(int)

    return pares


treino = construir_target(alunos, 2022, 2023)
teste = construir_target(alunos, 2023, 2024)

print(f"Treino (22→23): {len(treino)} alunos")
print(f"  Em risco: {treino['risco'].sum()} ({treino['risco'].mean() * 100:.1f}%)")
print(f"\nTeste  (23→24): {len(teste)} alunos")
print(f"  Em risco: {teste['risco'].sum()} ({teste['risco'].mean() * 100:.1f}%)")

In [ ]:
# Decomposição: quanto cada regra contribui?
def decompor(df, label):
    print(f"\n{label}:")
    for regra, desc in [
        ("r_queda", "Queda INDE >= 0.5"),
        ("r_regr", "Regressão de pedra"),
        ("r_evad", "Evasão"),
        ("r_p25", "Abaixo do p25 da fase"),
    ]:
        n = df[regra].sum()
        print(f"  {desc:25s}: {n:4d} ({n / len(df) * 100:.1f}%)")
    print(f"  {'Total em risco (OU)':25s}: {df['risco'].sum():4d} ({df['risco'].mean() * 100:.1f}%)")


decompor(treino, "Treino (22→23)")
decompor(teste, "Teste (23→24)")

In [ ]:
# Visualização do balanceamento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, label in zip(axes, [treino, teste], ["Treino (22→23)", "Teste (23→24)"]):
    counts = df["risco"].value_counts().sort_index()
    cores = ["#2a9d8f", "#e63946"]
    bars = ax.bar(["Sem risco", "Em risco"], counts.values, color=cores)
    for bar, v in zip(bars, counts.values):
        pct = v / counts.sum() * 100
        ax.text(bar.get_x() + bar.get_width() / 2, v + 10,
                f"{v}\n({pct:.1f}%)", ha="center", fontweight="bold")
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("Nº de alunos")
    ax.set_ylim(0, counts.max() * 1.2)

plt.suptitle("Balanceamento do target — 60/40, estável entre treino e teste",
             fontsize=13, y=1.02, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fe_balanceamento_target.png", dpi=150, bbox_inches="tight")
plt.show()

### 📊 Leitura do balanceamento

- **Treino: 59.7% em risco** (513 de 860)
- **Teste:  59.5% em risco** (603 de 1014)

O **balanceamento quase idêntico entre treino e teste** é um ótimo sinal — significa que o target é estável no tempo, e não vai ser um caso de "treinei num mundo e testo em outro". A regra do target é robusta.

**Classe majoritária é "em risco" (60%)** — nada de desbalanceamento extremo. Não precisamos de SMOTE ou técnicas de reamostragem; `class_weight='balanced'` nos modelos resolve se houver necessidade.

## 3. Feature engineering

Vamos criar 3 categorias de features:

### 🟦 Features de snapshot (valores brutos em t)
Os 7 indicadores principais + metadados básicos do aluno.

### 🟩 Features de engenharia
Derivadas dos insights das perguntas de negócio:
- `gap_iaa_ida` — captura o **Dunning-Kruger** da P4 (alunos que superestimam)
- `gap_ieg_ida` — captura "engajado mas sem performance"
- `ips_zscore_fase` — neutraliza a **anomalia do IPS em 2023** detectada na P5
- `inde_zscore_fase` — posição relativa do aluno dentro da sua fase
- `ranking_inde_fase` — percentil do aluno na fase (0 = pior, 1 = melhor)

### 🟨 Features de contexto
`anos_no_programa`, `fase`, `idade`, `pedra_nivel` (ordinal).

### ⚠️ Feature que NÃO vai entrar: IPP
O IPP não existe em 2022 (só aparece em 2023). Como nosso treino é 22→23, **incluir IPP geraria schema inconsistente** entre treino (sem IPP) e teste (com IPP). A decisão é **dropar IPP do modelo principal** — a gente documenta como limitação e pode criar uma v2 do modelo na evolução futura que usa IPP quando disponível.

In [ ]:
def engenharia_features(df):
    """
    Cria o DataFrame de features derivadas a partir dos dados de um ano (com
    target já calculado por construir_target).

    Schema FIXO — IPP excluído pra manter consistência treino/teste.
    """
    X = pd.DataFrame()

    # === 🟦 SNAPSHOT: indicadores e metadados no ano t ===
    for ind in ["inde", "ian", "ida", "ieg", "iaa", "ips", "ipv"]:
        X[ind] = df[ind]

    X["fase"] = df["fase"]
    X["idade"] = df["idade"]
    X["anos_no_programa"] = df["anos_no_programa"]
    X["ano_ingresso"] = df["ano_ingresso"]
    X["defasagem"] = df["defasagem"]

    # Gênero: 1 se menina, 0 caso contrário
    X["is_menina"] = (df["genero"].str.lower().str.contains("menina", na=False)).astype(int)

    # Pedra como ordinal (Quartzo=0, Topázio=3)
    X["pedra_nivel"] = df["pedra"].map(ORDEM_PEDRAS)

    # Notas brutas
    X["nota_mat"] = df["nota_mat"]
    X["nota_port"] = df["nota_port"]
    X["nota_ing"] = df["nota_ing"]
    X["tem_nota_ingles"] = df["tem_nota_ingles"].astype(int)

    # === 🟩 ENGENHARIA ===

    # Gap Dunning-Kruger (insight P4: 65% dos alunos superestimam)
    X["gap_iaa_ida"] = df["iaa"] - df["ida"]

    # Gap engajamento-desempenho ("engajado mas sem performance")
    X["gap_ieg_ida"] = df["ieg"] - df["ida"]

    # Média só de Mat+Port (robusta ao missing estrutural do inglês)
    X["media_notas_escolares"] = df[["nota_mat", "nota_port"]].mean(axis=1)

    # Z-score do IPS dentro da fase (neutraliza anomalia 2023 da P5)
    df_aux = df.copy()
    df_aux["ips_mean_fase"] = df_aux.groupby("fase")["ips"].transform("mean")
    df_aux["ips_std_fase"] = df_aux.groupby("fase")["ips"].transform("std")
    X["ips_zscore_fase"] = (df_aux["ips"] - df_aux["ips_mean_fase"]) / df_aux["ips_std_fase"]

    # Z-score do INDE dentro da fase (posição relativa)
    df_aux["inde_mean_fase"] = df_aux.groupby("fase")["inde"].transform("mean")
    df_aux["inde_std_fase"] = df_aux.groupby("fase")["inde"].transform("std")
    X["inde_zscore_fase"] = (df_aux["inde"] - df_aux["inde_mean_fase"]) / df_aux["inde_std_fase"]

    # Ranking percentil na fase (0 = pior, 1 = melhor)
    X["ranking_inde_fase"] = df.groupby("fase")["inde"].rank(pct=True)

    return X


X_train = engenharia_features(treino)
X_test = engenharia_features(teste)
y_train = treino["risco"].values
y_test = teste["risco"].values

print(f"✅ X_train: {X_train.shape}")
print(f"✅ X_test:  {X_test.shape}")
print(f"\nSchema idêntico? {list(X_train.columns) == list(X_test.columns)}")
print(f"\nFeatures ({X_train.shape[1]}):")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Diagnóstico de missing nas features
print("Missing por feature (treino):")
miss_train = (X_train.isna().sum() / len(X_train) * 100).round(1)
miss_train = miss_train[miss_train > 0].sort_values(ascending=False)
if len(miss_train) > 0:
    print(miss_train.to_string())
else:
    print("  (nenhum missing)")

print("\nMissing por feature (teste):")
miss_test = (X_test.isna().sum() / len(X_test) * 100).round(1)
miss_test = miss_test[miss_test > 0].sort_values(ascending=False)
if len(miss_test) > 0:
    print(miss_test.to_string())
else:
    print("  (nenhum missing)")

### 📊 Leitura do missing

- **Treino:** só `nota_ing` tem missing alto (67%) — é o missing estrutural do inglês (só algumas fases têm). Resolvido pela feature `tem_nota_ingles`.
- **Teste:** tem mais missing porque inclui os **universitários (fases 7-9)** que foram introduzidos em 2023/2024 sem avaliação completa. Também é informativo, não aleatório.

**Estratégia**: vamos deixar os missings como estão. **XGBoost e LightGBM lidam nativamente com NaN** — eles aprendem sozinhos a melhor direção quando encontram um missing. Pra LogReg/modelos lineares, vamos imputar com mediana dentro do pipeline. Não há imputação a priori no X.

## 4. Sanity check preliminar

Antes de partir pra modelagem séria, uma **Regressão Logística rápida** pra confirmar que:
1. O pipeline está funcionando end-to-end
2. As features carregam sinal (AUC > 0.5)
3. As features mais importantes fazem sentido conceitual

**Importante**: isso NÃO é o modelo final. É só um smoke test pra pegar bugs antes da Etapa 4B.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

pipe_smoke = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

pipe_smoke.fit(X_train, y_train)
pred = pipe_smoke.predict(X_test)
proba = pipe_smoke.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, proba)
print(f"🎯 ROC-AUC (sanity check): {auc:.3f}")
print(f"\nMatriz de confusão:")
cm = confusion_matrix(y_test, pred)
print(f"                pred 0  pred 1")
print(f"  real 0 (SR) :  {cm[0,0]:4d}    {cm[0,1]:4d}")
print(f"  real 1 (ER) :  {cm[1,0]:4d}    {cm[1,1]:4d}")
print(f"\n{classification_report(y_test, pred, target_names=['Sem risco', 'Em risco'])}")

In [ ]:
# Feature importance preliminar (coeficientes da LogReg)
coefs = pd.Series(
    pipe_smoke.named_steps["lr"].coef_[0],
    index=X_train.columns
).sort_values(key=abs, ascending=False)

print("Top 10 features por |coeficiente| (LogReg):")
for feat, val in coefs.head(10).items():
    direcao = "↑ risco" if val > 0 else "↓ risco"
    print(f"  {feat:25s}: {val:+.3f}  {direcao}")

# Visualização
fig, ax = plt.subplots(figsize=(10, 7))
top15 = coefs.head(15).iloc[::-1]
cores_coef = ["#e63946" if v > 0 else "#2a9d8f" for v in top15.values]
ax.barh(range(len(top15)), top15.values, color=cores_coef)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15.index)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 15 features (sanity check LogReg)\n"
             "Vermelho = aumenta risco, Verde = reduz risco", fontweight="bold")
ax.set_xlabel("Coeficiente")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fe_sanity_check_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Investigação: o coeficiente de pedra_nivel é contraintuitivo?
# Vamos ver a taxa de risco por pedra nos dois splits
print("Taxa de risco por pedra:")
print()
for label, df in [("Treino (22→23)", treino), ("Teste (23→24)", teste)]:
    print(f"{label}:")
    tabela = df.groupby("pedra")["risco"].agg(["mean", "count"]).round(3)
    tabela = tabela.reindex(["Quartzo", "Ágata", "Ametista", "Topázio"])
    print(tabela.to_string())
    print()

### 🔍 Leitura do sanity check

**ROC-AUC ≈ 0.71** — acima de 0.5 (aleatório) e abaixo de 0.85 (excelente). É um **bom sinal**: as features carregam informação real, mas ainda há espaço pra modelos mais sofisticados (GBMs tipicamente ganham 0.05-0.10 de AUC sobre LogReg em tabular).

**Features importantes batem com a narrativa das perguntas**:
- `inde_zscore_fase` — posição relativa do aluno na fase é fortíssimo preditor ✅
- `ieg`, `ian`, `ida` — os indicadores-chave identificados em P7
- `idade` — adolescentes e jovens adultos evadem mais (P10)

### 🧐 Achado contraintuitivo: o risco por pedra NÃO é monotônico

Olhando a tabela acima, perceba:

| Pedra | Risco (teste) |
|---|---|
| Quartzo | **87%** |
| Ágata | **70%** |
| Ametista | **47%** ⬇️ |
| **Topázio** | **51%** ⬆️ |

**Topázio tem mais risco que Ametista!** Isso parece errado à primeira vista, mas é um achado real:

1. **Alunos Topázio estão no teto do INDE** — matematicamente, só têm pra onde cair. Nossa regra (a) do target (queda ≥ 0.5) pega eles com mais facilidade.
2. **Topázio pode regredir de pedra** (pra Ametista) — Ametista também pode (pra Ágata), mas Topázio começa "mais exposto" a cair 1 nível.
3. Isso é **regressão à média estatística clássica**.

**Implicação pra modelagem**: a feature `pedra_nivel` como ordinal linear **perde esse padrão não-monotônico**. Modelos lineares (LogReg) vão interpretar como "mais alta a pedra = mais risco", que é parcialmente verdade mas perde a nuance. **GBMs (XGBoost, LightGBM) conseguem capturar a não-linearidade** via divisões em árvore — esperamos que eles ganhem do LogReg justamente nesse tipo de padrão.

⚠️ **Recall baixo pra classe 'em risco'**: o threshold default de 0.5 está gerando poucos positivos. Na Etapa 4B vamos **ajustar o threshold** pra maximizar recall (que é a métrica que importa pra ONG — perder aluno em risco é pior que falso positivo).

## 5. Export dos datasets

Salvamos 4 arquivos parquet em `data/processed/`:

- `X_train.parquet` / `y_train.parquet` — 860 alunos de 2022 com target de 2023
- `X_test.parquet` / `y_test.parquet` — 1014 alunos de 2023 com target de 2024

E também **metadados** pra análise de erros na Etapa 4D:
- `treino_metadata.parquet` / `teste_metadata.parquet` — RA, fase, pedra, regras individuais do target

In [ ]:
# Salvar datasets de modelagem
X_train.to_parquet(DATA_PROCESSED / "X_train.parquet", index=False)
X_test.to_parquet(DATA_PROCESSED / "X_test.parquet", index=False)
pd.Series(y_train, name="risco").to_frame().to_parquet(DATA_PROCESSED / "y_train.parquet", index=False)
pd.Series(y_test, name="risco").to_frame().to_parquet(DATA_PROCESSED / "y_test.parquet", index=False)

# Salvar metadados pra análise de erros
cols_meta = ["ra", "fase", "pedra", "inde", "idade", "genero",
             "risco", "r_queda", "r_regr", "r_evad", "r_p25"]
treino[cols_meta].to_parquet(DATA_PROCESSED / "treino_metadata.parquet", index=False)
teste[cols_meta].to_parquet(DATA_PROCESSED / "teste_metadata.parquet", index=False)

print("✅ Arquivos salvos em data/processed/:")
for arquivo in ["X_train", "y_train", "X_test", "y_test",
                "treino_metadata", "teste_metadata"]:
    path = DATA_PROCESSED / f"{arquivo}.parquet"
    if path.exists():
        print(f"  {arquivo}.parquet  ({path.stat().st_size / 1024:.1f} KB)")

# Sanity read-back
X_train_rb = pd.read_parquet(DATA_PROCESSED / "X_train.parquet")
assert X_train_rb.shape == X_train.shape, "Erro no read-back!"
print("\n✅ Read-back OK")

## ✅ Resumo da Etapa 4A

### Dataset pronto pra modelagem
- **24 features** (consistentes entre treino e teste)
- **Treino: 860 alunos** com features de 2022 e target de 2023 (60% em risco)
- **Teste: 1014 alunos** com features de 2023 e target de 2024 (60% em risco)

### Features criadas
- 🟦 **7 indicadores** + 8 metadados (snapshot em t)
- 🟩 **6 features de engenharia** (gaps, z-scores, rankings)
- 🟨 **3 features de contexto temporal** (anos no programa, fase, idade)

### Sanity check
- ROC-AUC de **~0.71** com LogReg simples — confirma que as features têm sinal
- Features mais importantes fazem sentido conceitual (posição relativa no INDE, pedra, engajamento, idade)

### O que fica pra Etapa 4B
- Treinar e comparar **3 modelos**: LogReg (baseline), XGBoost, LightGBM
- **Ajustar threshold** pra maximizar recall (critério de negócio: não perder alunos em risco)
- Avaliar com ROC-AUC, PR-AUC, F1, recall, matriz de confusão
- Selecionar modelo final e serializar pra `models/`

### Decisões que ficam documentadas como limitação
1. **IPP excluído** — inconsistência entre treino (sem IPP em 2022) e teste (com IPP em 2023)
2. **Apenas 2 pares (22→23, 23→24)** disponíveis — não há histórico mais longo
3. **Target inclui evasão** — diferente de outros trabalhos que preveem só queda de performance

### Próximo passo
**Etapa 4B — Modelagem tabular** (`04_modelagem_tabular.ipynb`)